In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)
import torch 

from auto_circuit.data import load_datasets_from_json
from auto_circuit.prune_algos.mask_gradient import mask_gradient_prune_scores
from auto_circuit.types import PruneScores
from auto_circuit.utils.graph_utils import patchable_model
from auto_circuit.utils.misc import repo_path_to_abs_path
from auto_circuit.visualize import draw_seq_graph

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import PreTrainedTokenizerFast, AutoTokenizer
import transformer_lens as tl
from transformer_lens import HookedTransformer, HookedTransformerConfig
import json


from auto_circuit.utils.ablation_activations import src_ablations
from auto_circuit.types import (
    AblationType,
    CircuitOutputs,
    PatchType,
    PruneScores,
)
from auto_circuit.utils.graph_utils import edge_counts_util, patchable_model, patch_mode
from auto_circuit.prune import run_circuits
from auto_circuit.metrics.prune_metrics.kl_div import measure_kl_div
from auto_circuit.utils.tensor_ops import correct_answer_proportion, correct_answer_greater_than_incorrect_proportion, batch_avg_answer_diff, batch_answer_diff_percents, correct_answer_greater_than_incorrect_proportion

%load_ext autoreload
%autoreload 2

In [ ]:
TOKENIZER_DIR  = "../model/wordlevel_tokenizer"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, add_bos_token=True)

MODEL = 'attn_only_rope'
# --- Load config ---
with open(f"../model/{MODEL}/config.json", "r") as f:
    cfg_dict = json.load(f)

# --- Fix dtype string back to actual torch dtype ---
if isinstance(cfg_dict.get("dtype"), str):
    cfg_dict["dtype"] = getattr(torch, cfg_dict["dtype"].replace("torch.", ""))

# --- Rebuild config and model ---
config = HookedTransformerConfig.from_dict(cfg_dict)
model = HookedTransformer(config)
model.load_state_dict(torch.load(f"../model/{MODEL}/model_weights.pth"))
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

model.set_tokenizer(tokenizer)

model.cfg.default_prepend_bos, model.cfg.tokenizer_prepends_bos

model.set_use_attn_result(True)
model.set_use_attn_in(True)
model.set_use_split_qkv_input(True)
if hasattr(model.cfg, "use_hook_mlp_in"):
    if model.cfg.use_hook_mlp_in is None:
        model.set_use_hook_mlp_in(True) 

model.eval()

for param in model.parameters():
    param.requires_grad = False

In [ ]:
path_last = repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/data/succession_augmented_last_big.json")
path_next = repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/data/succession_augmented_next_big.json")

train_loader, test_loader = load_datasets_from_json(
    model=model,
    path=path_next,
    device=device,
    prepend_bos=False,
    batch_size=16,
    train_test_size=(256, 256),
    tail_divergence=True,
)

auto_model = patchable_model(
    model,
    factorized=True,
    slice_output="last_seq",
    separate_qkv=True,
    kv_caches=(train_loader.kv_cache, test_loader.kv_cache),
    device=device,
)
for batch in test_loader:
    toks = batch.clean
    answers = batch.answers
    wrong_answers = batch.wrong_answers
    answers = [(answers[i], wrong_answers[i]) for i in range(len(answers))]

    answers = torch.tensor(answers, dtype=torch.long)
    answers = answers.to(device)

ablations = src_ablations(auto_model, toks, AblationType.RESAMPLE)
patch_edges = {
    "ResidStart->A0.1": 1.0,
    # "MLP0->MLP1": 1.0,
    # "A0.1->MLP0": 1.0
    # "MLP 2->A5.2.Q": 2.0,
    # "MLP 3->A5.2.Q": 1.0,
    # "A5.2->Resid End": 1.0,
}
ps: PruneScores = auto_model.circuit_prune_scores(edge_dict=patch_edges)

circuit_outs: CircuitOutputs = run_circuits(
    model=auto_model,
    dataloader=test_loader,
    test_edge_counts=edge_counts_util(auto_model.edges, prune_scores=ps),
    prune_scores=ps,
    patch_type=PatchType.EDGE_PATCH,
    ablation_type=AblationType.RESAMPLE,
)
# Average KL divergence between the full model and the circuits.
kl_divs = measure_kl_div(auto_model, test_loader, circuit_outs)
print("KL Divergence Results: ", kl_divs)

# Build circuit
prune_scores = auto_model.current_patch_masks_as_prune_scores()
fig = draw_seq_graph(auto_model, prune_scores)

In [ ]:
with patch_mode(auto_model, ablations, patch_edges):
    for batch in test_loader:
        patched_out = auto_model(batch.clean)

for batch in test_loader:
    clean_out = model(batch.clean)

proportion_correct = correct_answer_proportion(patched_out[:, -1, :], batch)
print("Proportion of correct answers after patching:", proportion_correct)

correct_answer_greater_than_incorrect_proportion_value = correct_answer_greater_than_incorrect_proportion(patched_out[:, -1, :], batch)
print("Proportion of correct answers greater than incorrect after patching:", correct_answer_greater_than_incorrect_proportion_value)

batch_avg_answer_diff_value = batch_avg_answer_diff(patched_out[:, -1, :], batch)
print("Average answer difference after patching:", batch_avg_answer_diff_value)

batch_avg_answer_diff_value_clean = batch_avg_answer_diff(clean_out[:, -1, :], batch)
print("Average answer difference full model:", batch_avg_answer_diff_value_clean)

batch_answer_diff_percents_value = batch_answer_diff_percents(patched_out[:, -1, :], clean_out[:, -1, :], batch)
print("Batch answer difference percents after patching:", batch_answer_diff_percents_value)

In [ ]:
auto_model = patchable_model(
    model,
    factorized=True,
    slice_output="last_seq",
    separate_qkv=True,
    device=device
)
for batch in test_loader:
    toks = batch.clean
    answers = batch.answers
    wrong_answers = batch.wrong_answers
    answers = [(answers[i], wrong_answers[i]) for i in range(len(answers))]

    answers = torch.tensor(answers, dtype=torch.long)
    answers = answers.to(device)
    # print(answers)

ablations = src_ablations(auto_model, toks, AblationType.RESAMPLE)

patch_edges = [
    "A0.1->A1.0.Q",
    "A1.0->Resid End",
]
target_edge = "Resid Start->A0.1.Q"
assert any(target_edge in edge_list for edge_list in auto_model.edge_dict.values())
with patch_mode(auto_model, ablations, patch_edges):
    patched_out = auto_model(toks)

from utils.direct_logit_attribution import logits_to_ave_logit_diff
ave_logit_diff = logits_to_ave_logit_diff(patched_out, answer_tokens=answers, per_prompt=False)
print("Average Logit Difference:", ave_logit_diff)

In [ ]:
import itertools
target_edge_str = "Resid Start->A0.1.Q"

all_edges = list(itertools.chain.from_iterable(auto_model.edge_dict.values()))
all_edge_strs = [str(edge) for edge in all_edges]

assert target_edge_str in all_edge_strs, f"{target_edge_str} not found"